# Introduction


# Part 1: Creating a Class

Create spark session and import spark_data_check module

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

from pyspark.sql import functions as F
from spark_data_check import SparkDataCheck

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 17:21:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/26 17:21:29 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


### Class Method: read_csv()

The read_csv() class method uses spark.read.load() to create an instance of the SparkDataCheck class while reading in a csv. We will use it to create an instance of SparkDataCheck and read in the air.csv file.

In [2]:
air = SparkDataCheck.read_csv(spark, file_path = 'air.csv')

# drop extra index column
air.df = air.df.drop("_c0")

# replace periods in column names with underscores to avoid errors
new_cols = [c.replace(".", "_") for c in air.df.columns]
air.df = air.df.toDF(*new_cols)

# show first 5 rows of data frame
air.df.show(5)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-26 21:00:00|   2.2|       1376|      80|     9.2|        

Print air data schema to look at column names and types

In [34]:
air.df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



Now that we have read in the data, we can use it to test each of the methods in the SparkDataCheck class. 

### Method: check_numeric_col()

For the first method, check_numeric_col(), we will first test it by providing 'CO(GT)' as the numeric column.

The function has worked as intended in this case, we can see all of the values were correctly assigned 'true' or 'false' based on whether they are between the provided bounds. 

In [14]:
air.check_numeric_col(column = 'CO(GT)', lower = 0, upper = 10)
air.df.select('CO(GT)', 'within_num_range').show()

+------+----------------+
|CO(GT)|within_num_range|
+------+----------------+
|   2.6|            true|
|   2.0|            true|
|   2.2|            true|
|   2.2|            true|
|   1.6|            true|
|   1.2|            true|
|   1.2|            true|
|   1.0|            true|
|   0.9|            true|
|   0.6|            true|
|-200.0|           false|
|   0.7|            true|
|   0.7|            true|
|   1.1|            true|
|   2.0|            true|
|   2.2|            true|
|   1.7|            true|
|   1.5|            true|
|   1.6|            true|
|   1.9|            true|
+------+----------------+
only showing top 20 rows


Now we will test that the error messages work properly.

First, try providing no bounds. We will test the method on the numeric column 'T' this time.

The result is as expected, with an error message being printed saying to provide at least one bound. 

In [15]:
air.check_numeric_col(column = 'T')

Error: Must provide at least one of 'upper' or 'lower'


Now we will test the method by only providing one bound.

This test worked as expected, with the column only being compared to the provided lower bound. 

In [20]:
air.check_numeric_col(column = 'T', lower = 10)
air.df.select('T', 'within_num_range').show()

+----+----------------+
|   T|within_num_range|
+----+----------------+
|13.6|            true|
|13.3|            true|
|11.9|            true|
|11.0|            true|
|11.2|            true|
|11.2|            true|
|11.3|            true|
|10.7|            true|
|10.7|            true|
|10.3|            true|
|10.1|            true|
|11.0|            true|
|10.5|            true|
|10.2|            true|
|10.8|            true|
|10.5|            true|
|10.8|            true|
|10.5|            true|
| 9.5|           false|
| 8.3|           false|
+----+----------------+
only showing top 20 rows


Lastly, we will test the method by providing a non-numeric column. The 'Time' column is a timestamp, so we will use it for this test.

The result is as expected, with an error message printing telling us to provide a numeric column.

In [12]:
air.check_numeric_col(column = 'Time', lower = 0, upper = 10)

Error: Column must of type float, int, longint, bigint, double, or integer


### Method: check_string_col

Now we will test the method check_string_col(). This method accepts a column name and a set of levels and returns a data frame with a column added indicating whether the values of the column are within the user specified levels. 

First, we will test the method using the 'Date' column, which is a string.

The result is as expected - a boolean column called 'within_level' was added and we can see that the values of within_level are correct.

In [17]:
air.check_string_col(column = 'Date', levels = ['3/10/2004', '3/11/2004'])
air.df.select('Date', 'within_level').show()

+---------+------------+
|     Date|within_level|
+---------+------------+
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
|3/11/2004|        true|
+---------+------------+
only showing top 20 rows


Now let's see what we get when we provide levels that are not in the provided column.

All of the 'within_level' values are false, so the method worked correctly.

In [18]:
air.check_string_col(column = 'Date', levels = ['0', '1', '2'])
air.df.select('Date', 'within_level').show()

+---------+------------+
|     Date|within_level|
+---------+------------+
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 20 rows


Next we can test the method by providing a level that is in the Date column and a level that is not in the date column.

All of the rows where Date = '3/10/2004' are true for the within_level column and all of the other rows are false for within_level, so this worked correctly.

In [19]:
air.check_string_col(column = 'Date', levels = ['3/10/2004', '1'])
air.df.select('Date', 'within_level').show()

+---------+------------+
|     Date|within_level|
+---------+------------+
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/10/2004|        true|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 20 rows


Lastly, we will test the method by providing a non-string column to the function.

This correctly returned an error message stating that the provided column must be a string column.

In [9]:
air.check_string_col(column = 'T', levels = [1, 2, 3])

Error: Column type must be string.


### Method: check_missing_values()

Next, we wil test the check_missing_values() method. This method checks a column for missing values and returns the modified data frame with a boolean column indicating whether the values in a column are missing. 

Let's first test the function on a numeric column. 

The 'is_missing' column was added properly and correctly show that none of these values of 'AH' are missing. 

In [ ]:
air.check_missing_values(column = 'AH')
air.df.select('AH', 'is_missing').show()

Let's now test this method on a different type of column. We will test it on the 'Time' column, which is a timestamp.

Again, the result is as expected- none of the values are missing and all of the 'is_missing' values say false.

In [ ]:
air.check_missing_values(column = 'Time')
air.df.select('Time', 'is_missing').show()

In order to confirm this method works as intended, we want to test it on a column that has null values. Let's check the number of null values in each column to see which column we can test this method on.

In [ ]:
air.df.select([
    F.sum(F.when(F.col(f"`{c}`").isNull(), 1).otherwise(0)).alias(c)
    for c in air.df.columns
]).show()

Currently, none of the columns have null values, so we can't confirm that the method correctly identifies null values. 

However, we know from the first project that the variables 'C6H6(GT)' and 'CO(GT)' contains missing values, which are coded as '-200'. We can reassign these values as nulls and then test our check_missing_values() method. 

Then we can rerun the code to check the number of missing values per column to make sure this worked.

CO(GT) now has 1683 missing values and C6H6(GT) has 366 missing values, so we can test our method on these columns.

In [ ]:
# columns to reassign values to missing for
cols = ['CO(GT)', 'C6H6(GT)']

# replace values for both columns
for col in cols:
    air.df = air.df.withColumn(
        col,
        F.when(F.col(col) == -200, None).otherwise(F.col(col))
    )

# get null value counts after replacing values
air.df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in air.df.columns
]).show()

Now we can test the method on these two columns. 

In this output, there is one null value for CO(GT), and we can see that check_missing_values correctly assigned 'false' to this row.

In [ ]:
air.check_missing_values(column = 'CO(GT)')
air.df.select('CO(GT)', 'is_missing').show()

Now let's test the method on C6H6(GT).

This time, we will return the count of true and false values from the 'is_missing' column to make sure the number of true values aligns with the number of nulls in 'C6H6(GT)'. 

This returned 366, which is the same number of null values in the column.

In [ ]:
air.check_missing_values(column = 'C6H6(GT)')
air.df.select('C6H6(GT)', 'is_missing').groupby('is_missing').count().show()

### Method: summarize_min_max()

This method takes in a numeric column and reports the minimum and maximum values of the column. A grouping column can optionally be provided. If no column name is provided, all numeric columns will be summarized.

First, let's test the method using a numeric column with no grouping column.

The output is a pandas data frame with 'CO(GT)_min' and 'CO(GT)_max' columns, as expected.

In [7]:
air.summarize_min_max(column = 'CO(GT)')

,CO(GT)_min,CO(GT)_max
0,0.1,11.9


Now lets test the method using 'Date' as the grouping column and 'C6H6(GT)' as the numeric column to summarize.

I used sort_values() to sort the result by the Date column to make it easier to read.

This returned the minimum and maximum values of 'C6H6(GT)' for each value of 'Date' as expected.

In [8]:
air.summarize_min_max(column = 'C6H6(GT)', group_col = 'Date').sort_values('Date')

,Date,C6H6(GT)_max,C6H6(GT)_max
350,1/1/2005,3.0,16.6
308,1/10/2005,1.3,21.4
117,1/11/2005,2.1,26.5
36,1/12/2005,4.2,28.4
253,1/13/2005,3.2,26.6
...,...,...,...
172,9/5/2004,1.9,30.3
39,9/6/2004,1.0,13.5
382,9/7/2004,2.7,22.0
131,9/8/2004,10.6,34.5


Now let's try passing no numeric column and no grouping column to the method. It should find all numeric columns in the data frame and find the minimum and maximum values for each one.

In [ ]:
air.summarize_min_max()

Next, we can try passing a column to group by, but no numeric column. The method should find all numeric columns in the data frame and summarize the minimum and maximum values of each, grouped by the given grouping column. We will again use 'Date' as the grouping column.

In [ ]:
air.summarize_min_max(group_col = 'Date').sort_values('Date')

### Method: get_value_counts()

The get_value_counts() method takes in one or two string columns and gets the counts. If one column is provided, it will count the number of instances of each value of the column. If two columns are provided, it will report the counts for each combination of the two columns.

Since the air data frame only has one string column, 'Date', we need to make a few more so that we can properly test this method.

We will split the date column into three parts to get a day column, month column, and year column. 

In [ ]:
air.df = (
    air.df
    .withColumn('Month', F.split(F.col('Date'), '/').getItem(0))
    .withColumn('Day',   F.split(F.col('Date'), '/').getItem(1))
    .withColumn('Year',  F.split(F.col('Date'), '/').getItem(2))
)
air.df.show(5)

Now we need to test the get_value_counts() method. First we will test it using just one string column.

From the output we can see that the method correctly reported the number of observations for each value of 'Date'.

In [ ]:
air.get_value_counts(column1 = 'Date')

Now let's try providing two columns to the method. We will use 'Month' as column 1 and 'Year' as column 2. The result should be a data frame containing the number of instances of each combination of 'Month' and 'Year'

In [ ]:
air.get_value_counts(column1 = 'Month', column2 = 'Year')

Next we can try passing a numeric column to the method.

This correctly resulted in an error stating that the column must be numeric.

In [ ]:
air.get_value_counts(column1 = 'T')

Now, we will make sure we get an error if we try to pass a numeric column to the 'column2' argument. Let's test the method using 'Month' for the first column and 'T' for the second column.

This correctly output an error stating that the columns need to be numeric.

In [ ]:
air.get_value_counts(column1 = 'Month', column2 = 'T')

### Class Method: read_pandas()

The read_pandas() method creates an instance of the SparkDataCheck class from a pandas data frame.

Let's read in air.csv with pandas and then use our method to create an instance of SparkDataCheck.

In [ ]:
import pandas as pd

# read in air.csv with pandas
pandas_df = pd.read_csv('air.csv', index_col = 0)

# use method to create instance of class
air_pd = SparkDataCheck.read_pandas(spark, pandas_df)

Let's use one of our methods on this data frame. We will test the 'check_numeric_col()' method using the 'CO(GT)' column. 

In [ ]:
air_pd.check_numeric_col('CO(GT)', lower = 0, upper = 10)
air_pd.df.select('CO(GT)', 'within_num_range').show()

# Part 2: NFL Data Analysis

For part 2, we will do some analysis on weekly NFL data using Spark SQL and pandas-on-spark.

We will start by doing the analysis using only pandas-on-spark. 

First, we need to import some packages and read in the data set.

In [2]:
import pyspark.pandas as ps

ps.set_option('compute.fail_on_ansi_mode', False)
import warnings
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning
warnings.simplefilter("ignore", category=PandasAPIOnSparkAdviceWarning)

Read in the weekly NFL data using PySpark and look at the first 5 rows

In [3]:
nfl = ps.read_csv("weekly_nfl_data.csv")
nfl.head()

26/03/26 19:39:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Let's look at the column names in the data frame.

In [4]:
nfl.columns.tolist()

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

We are only interested in seasons 2005 to 2023, the position 'QB', and the regular season. Let's filter the data accordingly.

In [5]:
nfl_subset = nfl[
    (nfl["position"] == "QB") &
    (nfl["season_type"] == "REG") &
    (nfl["season"].between(2005, 2023))
]

# look at the first few rows of data after subsetting
nfl_subset.head(3)

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
29406,00-0000722,None,Tony Banks,QB,QB,None,HOU,2005,17,REG,SF,14,25,173.0,1,2.0,0.0,-0.0,0,0,0.0,0.0,7.0,-3.693779,0,0.0,0.032036,2,-2.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,6.72,6.72
29426,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,9,REG,GB,9,16,65.0,0,1.0,1.0,6.0,1,0,0.0,0.0,4.0,-6.609788,0,0.0,0.025249,8,14.0,0,0.0,0.0,1.0,-1.614163,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,2.00,2.00
29427,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,10,REG,CLE,13,19,150.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,7.0,5.193222,0,0.0,0.171917,3,16.0,1,0.0,0.0,2.0,0.737467,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,13.60,13.60


We are only interested in the columns player_display_name, season, week, completions, attempts, passing?_yards, passing_tds, and interceptions, so we will subset the nfl_subset dataframe to only include these columns.

In [6]:
nfl_subset = nfl_subset[
    [
        "player_display_name",
        "season",
        "week",
        "completions",
        "attempts",
        "passing_yards",
        "passing_tds",
        "interceptions"
    ]
]

# look at the first few rows of data after subsetting
nfl_subset.head(3)

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0


Now we want to find the sum of the week, completions, attempts, passing_yards, passing_tds, and interceptions columns for each player and season combination. To do this, we will group by player and season and use .agg() to calculate the metrics for the remaining columns.

In [7]:
nfl_summary = (
    # group by player and season
    nfl_subset.groupby(["player_display_name", "season"])
      # compute sum and mean of remaining columns
      .agg({
          "completions": ["sum", "mean"],
          "attempts": ["sum", "mean"],
          "passing_yards": ["sum", "mean"],
          "passing_tds": ["sum", "mean"],
          "interceptions": ["sum", "mean"]
      })
)

# look at the first few rows of the summarized data
nfl_summary.head(3)

completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jake Delhomme       2006           263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154
Jake Plummer        2005           277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500
Matt Schaub         2006            18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000

Next we want to create two new variables using this summarized data:
- completion_percentage = (sum of completions)/(sum of attempts)
- td_int_ratio = (sum passing tds)/(sum interceptions)

Before we can do this, we need to flatten the summarized data frame to get rid of the multilevel columns. We also need to turn the player and season columns into regular columns rather than index levels.

In [8]:
# rename columns to include the original column name and the metric
nfl_summary.columns = [
    f"{col[0]}_{col[1]}" for col in nfl_summary.columns
]

# turn player and season into regular columns instead of index
nfl_summary = nfl_summary.reset_index()
nfl_summary.head(3)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000


Now that we have flattened the data frame, we can compute our new variables.

First we will calculate completion percentage.

The completion percentage calculation uses the column 'attempts_sum' as the denominator and the td_int_ratio calculation uses the column 'interceptions_sum' as the denominator. Both of these columns have zero values, so we need to replace those with null values to avoid getting an error due to trying to divide by zero. 

In [17]:
# calculate completion percentage
nfl_summary["completion_percentage"] = (
    nfl_summary["completions_sum"] / nfl_summary["attempts_sum"]
)

# replace values of 0 for interceptions_sum with None to avoid divide by zero error
nfl_summary[["attempts_sum", "interceptions_sum"]] = nfl_summary[["attempts_sum", "interceptions_sum"]].replace(0, None)

# calculate td_int_ratio
nfl_summary["td_int_ratio"] = (
    nfl_summary["passing_tds_sum"] / nfl_summary["interceptions_sum"]
)

nfl_result = nfl_summary

# view the first few rows of the result data frame
nfl_result.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/frame.py:6420: FutureWarning: DataFrame.replace without 'value' and with non-dict-like 'to_replace' is deprecated and will raise in a future version. Explicitly specify the new values instead.
  warnings.warn(


,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,0.666667,0.500000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,NaN,0.000000,0.609756,NaN


Now that we have these metrics and calculated variables, we want to do some more subsetting.

Let's start by finding the player/season combinations where the sum of attempts is at least 50. 

In [24]:
nfl_result[nfl_result["attempts_sum"] >= 50]

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,NaN,0.000000,0.609756,NaN
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,NaN,0.000000,0.576923,NaN
7,Eli Manning,2006,301,18.812500,522,32.625000,3244.0,202.750000,25,1.562500,18.0,1.125000,0.576628,1.388889
9,Mark Brunell,2006,162,16.200000,261,26.100000,1789.0,178.900000,8,0.800000,4.0,0.400000,0.620690,2.000000
10,Shaun Hill,2007,54,18.000000,80,26.666667,501.0,167.000000,5,1.666667,2.0,0.666667,0.675000,2.500000
11,Aaron Brooks,2006,110,13.750000,192,24.000000,1105.0,138.125000,3,0.375000,8.0,1.000000,0.572917,0.375000
12,Brett Favre,2006,343,21.437500,613,38.312500,3885.0,242.812500,18,1.125000,18.0,1.125000,0.559543,1.000000


Now let's find the 40 player/season combinations with the highest completion percentage

In [23]:
nfl_result.sort_values(
    by = "completion_percentage",
    ascending = False
).head(40)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
57,Anthony Wright,2006,3,1.500000,3,1.500000,31.0,15.500000,0,0.000000,NaN,0.000000,1.000000,NaN
330,Matt Gutierrez,2007,1,0.250000,1,0.250000,15.0,3.750000,0,0.000000,NaN,0.000000,1.000000,NaN
341,Matt Gutierrez,2009,1,1.000000,1,1.000000,3.0,3.000000,0,0.000000,NaN,0.000000,1.000000,NaN
347,Brad Smith,2009,1,0.090909,1,0.090909,27.0,2.454545,0,0.000000,NaN,0.000000,1.000000,NaN
422,Billy Volek,2010,1,0.333333,1,0.333333,8.0,2.666667,0,0.000000,NaN,0.000000,1.000000,NaN
474,Jordan Palmer,2010,3,3.000000,3,3.000000,18.0,18.000000,0,0.000000,NaN,0.000000,1.000000,NaN
479,Dennis Dixon,2008,1,1.000000,1,1.000000,3.0,3.000000,0,0.000000,NaN,0.000000,1.000000,NaN
538,Colt McCoy,2013,1,0.333333,1,0.333333,13.0,4.333333,0,0.000000,NaN,0.000000,1.000000,NaN
550,Tyrod Taylor,2011,1,0.500000,1,0.500000,18.0,9.000000,0,0.000000,NaN,0.000000,1.000000,NaN
563,Trent Edwards,2012,2,2.000000,2,2.000000,14.0,14.000000,0,0.000000,NaN,0.000000,1.000000,NaN


Lastly, we will find the 40 player/season combinations with the highest td_int_ratio

In [22]:
nfl_result.sort_values(
    by = "td_int_ratio",
    ascending = False
).head(40)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
763,Tom Brady,2016,291,24.250000,432,36.000000,3554.0,296.166667,28,2.333333,2.0,0.166667,0.673611,14.000000
639,Nick Foles,2013,203,15.615385,317,24.384615,2891.0,222.384615,27,2.076923,2.0,0.153846,0.640379,13.500000
618,Josh McCown,2013,149,18.625000,224,28.000000,1829.0,228.625000,13,1.625000,1.0,0.125000,0.665179,13.000000
989,Aaron Rodgers,2018,372,23.250000,597,37.312500,4442.0,277.625000,25,1.562500,2.0,0.125000,0.623116,12.500000
200,Damon Huard,2006,148,14.800000,244,24.400000,1878.0,187.800000,11,1.100000,1.0,0.100000,0.606557,11.000000
1262,Aaron Rodgers,2020,372,23.250000,526,32.875000,4299.0,268.687500,48,3.000000,5.0,0.312500,0.707224,9.600000
1111,Aaron Rodgers,2021,366,22.875000,531,33.187500,4115.0,257.187500,37,2.312500,4.0,0.250000,0.689266,9.250000
286,Tom Brady,2010,324,20.250000,492,30.750000,3900.0,243.750000,36,2.250000,4.0,0.250000,0.658537,9.000000
20,Jake Delhomme,2007,55,18.333333,86,28.666667,617.0,205.666667,8,2.666667,1.0,0.333333,0.639535,8.000000
695,Aaron Rodgers,2014,341,21.312500,520,32.500000,4381.0,273.812500,38,2.375000,5.0,0.312500,0.655769,7.600000
